In [1]:
%run_nb spark-start --data-format iceberg

Args: Namespace(data_format='iceberg', port_offset=2) - unknown_args: []
Spark version: 4.1.2, Driver memory: 16g, Executor memory: 8g, Service: jupyter-spark-4.1, Data format: iceberg
Spark packages: org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0
Spark extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Spark catalog configs: {'spark.sql.catalog.local': 'org.apache.iceberg.spark.SparkCatalog', 'spark.sql.catalog.local.type': 'hadoop', 'spark.sql.catalog.local.warehouse': 'file:///home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse'}


,catalog
0,spark_catalog


Version,4.1.2
Master,local[2]
AppName,main


               total        used        free      shared  buff/cache   available
Mem:            62Gi        18Gi        14Gi       507Mi        31Gi        44Gi
Swap:          8.0Gi          0B       8.0Gi


In [5]:
path="/home/jovyan/work/data/kaggle/datasets/stefanoleone992/fifa-23-complete-player-dataset"
!ls -la {path}
pd.set_option("display.max_colwidth", None) # Disable collumn truncation

total 5799848
drwxr-xr-x. 1 jovyan users        318 Jul 22 16:27  .
drwxr-xr-x. 1 jovyan  1000         62 Jul 15 20:37  ..
drwxr-xr-x. 1 jovyan users         16 Jul 15 20:39  .complete
-rw-r--r--. 1 jovyan users       5356 Jul 15 20:38  female_coaches.csv
-rw-r--r--. 1 jovyan users    1685124 Jul 15 20:38 'female_players (legacy).csv'
-rw-r--r--. 1 jovyan users   94212088 Jul 15 20:38  female_players.csv
-rw-r--r--. 1 jovyan users    2214250 Jul 15 20:38  female_teams.csv
-rw-r--r--. 1 jovyan users     132879 Jul 15 20:38  male_coaches.csv
-rw-r--r--. 1 jovyan users   90933390 Jul 15 20:38 'male_players (legacy).csv'
-rw-r--r--. 1 jovyan users 5637100640 Jul 15 20:39  male_players.csv
-rw-r--r--. 1 jovyan users  112744779 Jul 15 20:39  male_teams.csv
time: 115 ms (started: 2026-07-22 20:25:48 +00:00)


In [6]:
csv_file = Path(path) / "male_players.csv"

time: 299 μs (started: 2026-07-22 20:25:51 +00:00)


In [7]:
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 287 μs (started: 2026-07-22 20:25:52 +00:00)


In [ ]:
%sql DROP TABLE local.fifa.male_players_partition_fifa_update_date

# Ingest partitioning by "fifa_update_date"

In [22]:
def ingest_table(table:str, partition: str):
    table = f"{table}_partition_{partition}"
    print(f"Table name: {table}")
    if spark.catalog.tableExists(table):
        print(f"Reading existing Iceberg table: {table}")
        return spark.table(table)

    print(f"Creating partitioned Iceberg table: {table}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(str(csv_file))
    )

    (
        df.writeTo(table)
        .using("iceberg")
        .partitionedBy(F.col(partition))
        .create()
    )

    return spark.table(table)
    

time: 287 μs (started: 2026-07-22 20:28:23 +00:00)


In [24]:
df = ingest_table("local.fifa.male_players", "fifa_update_date")

Table name: local.fifa.male_players_partition_fifa_update_date
Reading existing Iceberg table: local.fifa.male_players_partition_fifa_update_date
time: 8.85 ms (started: 2026-07-22 20:30:52 +00:00)


# Number and size of files
 - Issue: 2MB small file size problem. Over-partitioning
 - Better size: 128MB to 512MB

In [25]:
%%sql
SELECT count(*) as count
FROM local.fifa.male_players_partition_fifa_update_date.data_files;


SELECT
    file_path,
    file_format,
    record_count,
    format_number(file_size_in_bytes, 0) AS file_size_in_bytes
FROM local.fifa.male_players_partition_fifa_update_date.data_files;

,count
0,553


,file_path,file_format,record_count,file_size_in_bytes
0,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2022-07-07/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00001.parquet,PARQUET,18642,"2,208,346"
1,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2018-11-01/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00013.parquet,PARQUET,18191,"2,150,022"
2,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2017-08-10/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00022.parquet,PARQUET,17597,"2,053,100"
3,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2017-01-30/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00027.parquet,PARQUET,17469,"2,036,838"
4,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2021-06-22/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00008.parquet,PARQUET,18388,"2,184,093"
5,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2017-08-14/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00020.parquet,PARQUET,17597,"2,053,092"
6,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2019-03-14/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00011.parquet,PARQUET,18070,"2,139,690"
7,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2019-01-07/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00014.parquet,PARQUET,18075,"2,137,003"
8,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2018-02-26/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00017.parquet,PARQUET,17921,"2,107,986"
9,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/fifa/male_players_partition_fifa_update_date/data/fifa_update_date=2015-03-06/00000-132-ee46c548-8ed5-4905-a09b-12e163456dfe-0-00039.parquet,PARQUET,16297,"1,841,728"


time: 251 ms (started: 2026-07-22 20:31:07 +00:00)


In [27]:
%%sql
SELECT *
FROM  local.fifa.male_players_partition_fifa_update_date
WHERE fifa_update_date="2014-11-28"
LIMIT 3


,player_id,player_url,fifa_version,fifa_update,fifa_update_date,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,dob,height_cm,weight_kg,league_id,league_name,league_level,club_team_id,club_name,club_position,club_jersey_number,club_loaned_from,club_joined_date,club_contract_valid_until_year,nationality_id,nationality_name,nation_team_id,nation_position,nation_jersey_number,preferred_foot,weak_foot,skill_moves,international_reputation,work_rate,body_type,real_face,release_clause_eur,player_tags,player_traits,pace,shooting,passing,dribbling,defending,physic,attacking_crossing,attacking_finishing,attacking_heading_accuracy,attacking_short_passing,attacking_volleys,skill_dribbling,skill_curve,skill_fk_accuracy,skill_long_passing,skill_ball_control,movement_acceleration,movement_sprint_speed,movement_agility,movement_reactions,movement_balance,power_shot_power,power_jumping,power_stamina,power_strength,power_long_shots,mentality_aggression,mentality_interceptions,mentality_positioning,mentality_vision,mentality_penalties,mentality_composure,defending_marking_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_diving,goalkeeping_handling,goalkeeping_kicking,goalkeeping_positioning,goalkeeping_reflexes,goalkeeping_speed,ls,st,rs,lw,lf,cf,rf,rw,lam,cam,ram,lm,lcm,cm,rcm,rm,lwb,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk,player_face_url
0,158023,/player/158023/lionel-messi/150014,15,14,2014-11-28,L. Messi,Lionel Andrés Messi Cuccittini,CF,93,95,100500000,550000,27,1987-06-24,170,72,53,La Liga,1,241,FC Barcelona,CF,10,None,2004-07-01,2018,52,Argentina,1369,RW,10,Left,3,4,5,Medium/Low,Normal (170-),Yes,NaN,"#Speedster, #Dribbler, #FK Specialist, #Acrobat, #Clinical Finisher, #Complete Forward","Finesse Shot, Speed Dribbler (AI), One Club Player, Team Player",93,89,86,96,27,63,84,94,71,89,85,96,89,90,76,96,96,90,94,94,95,80,73,77,60,88,48,22,92,90,76,NaN,25,21,20,6,11,15,14,8,NaN,89+3,89+3,89+3,92+3,90+3,90+3,90+3,92+3,92+3,92+3,92+3,90+3,79+3,79+3,79+3,90+3,62+3,62+3,62+3,62+3,62+3,54+3,45+3,45+3,45+3,54+3,15+3,https://cdn.sofifa.net/players/158/023/15_120.png
1,20801,/player/20801/c-ronaldo-dos-santos-aveiro/150014,15,14,2014-11-28,Cristiano Ronaldo,Cristiano Ronaldo dos Santos Aveiro,"LW, LM, ST",92,92,79000000,375000,29,1985-02-05,185,80,53,La Liga,1,243,Real Madrid,LW,7,None,2009-07-01,2018,38,Portugal,1354,LW,7,Right,4,5,5,High/Low,Normal (185+),Yes,NaN,"#Speedster, #Dribbler, #Distance Shooter, #Acrobat, #Clinical Finisher, #Complete Forward","Power Free-Kick, Flair, Long Shot Taker (AI), Speed Dribbler (AI)",93,93,81,91,32,79,83,95,86,82,87,93,88,79,72,92,91,94,93,90,63,94,94,89,79,93,63,24,91,81,85,NaN,22,31,23,7,11,15,14,11,NaN,91+1,91+1,91+1,89+3,91+1,91+1,91+1,89+3,89+3,89+3,89+3,87+3,77+3,77+3,77+3,87+3,63+3,63+3,63+3,63+3,63+3,57+3,52+3,52+3,52+3,57+3,16+3,https://cdn.sofifa.net/players/020/801/15_120.png
2,9014,/player/9014/arjen-robben/150014,15,14,2014-11-28,A. Robben,Arjen Robben,"RM, LM, RW",90,90,54500000,275000,30,1984-01-23,180,80,19,Bundesliga,1,21,FC Bayern München,SUB,10,None,2009-08-28,2017,34,Netherlands,105035,RW,11,Left,2,4,5,High/Low,Normal (170-185),Yes,NaN,"#Speedster, #Dribbler, #Distance Shooter, #Acrobat","Diver, Injury Prone, Avoids Using Weaker Foot, Selfish, Long Shot Taker (AI), Speed Dribbler (AI), Chip Shot (AI)",93,86,83,93,32,64,80,85,50,86,86,93,85,83,76,92,93,93,93,91,91,86,61,78,65,90,47,39,89,84,80,NaN,29,26,26,10,8,11,5,15,NaN,84+3,84+3,84+3,89+1,87+3,87+3,87+3,89+1,89+1,89+1,89+1,87+3,79+3,79+3,79+3,87+3,64+3,64+3,64+3,64+3,64+3,55+3,46+3,46+3,46+3,55+3,14+3,https://cdn.sofifa.net/players/009/014/15_120.png


time: 140 ms (started: 2026-07-22 20:32:34 +00:00)


In [49]:
table = "local.fifa.male_players_partition_fifa_update_date"

time: 117 μs (started: 2026-07-22 20:49:22 +00:00)


In [50]:
df_select_filter = (
    spark.table(table)
    .filter(F.col("fifa_update_date") == "2014-11-28")
    .select("fifa_update_date", "club_name")
)

time: 11.2 ms (started: 2026-07-22 20:49:24 +00:00)


In [51]:
print(
    df_select_filter
    ._jdf
    .queryExecution()
    .executedPlan()
    .toString()
)

*(1) ColumnarToRow
+- BatchScan local.fifa.male_players_partition_fifa_update_date[fifa_update_date#3165, club_name#3181] IcebergScan(table=local.fifa.male_players_partition_fifa_update_date, schemaId=0, snapshotId=2327944090508763297, branch=null, filters=fifa_update_date IS NOT NULL, fifa_update_date = 16402, runtimeFilters=, groupedBy=) RuntimeFilters: []

time: 21.1 ms (started: 2026-07-22 20:49:25 +00:00)


In [54]:
all_files = spark.table(table).inputFiles()
filtered_files = (
    spark.table(table)
    .filter(F.col("fifa_update_date") == F.lit("2014-11-28"))
    .inputFiles()
)
print("All files:", len(all_files))
print("Filtered files:", len(filtered_files))

All files: 0
Filtered files: 0
time: 33.2 ms (started: 2026-07-22 20:49:43 +00:00)


# PushedFilters doesn't appear in the dataframe explain!
Partition pruning is the main optimization. The absence of the literal PushedFilters label does not mean the filter was ignored.

In [40]:
df_select_filter.explain("formatted")

== Physical Plan ==
* ColumnarToRow (2)
+- BatchScan local.fifa.male_players_partition_fifa_update_date (1)


(1) BatchScan local.fifa.male_players_partition_fifa_update_date
Output [2]: [fifa_update_date#3037, club_name#3053]
IcebergScan(table=local.fifa.male_players_partition_fifa_update_date, schemaId=0, snapshotId=2327944090508763297, branch=null, filters=fifa_update_date IS NOT NULL, fifa_update_date = 16402, runtimeFilters=, groupedBy=)

(2) ColumnarToRow [codegen id : 1]
Input [2]: [fifa_update_date#3037, club_name#3053]


time: 19.9 ms (started: 2026-07-22 20:40:46 +00:00)


In [47]:
df_select_filter.explain("extended")

== Parsed Logical Plan ==
'Project ['fifa_update_date, 'club_name]
+- Filter (fifa_update_date#3037 = cast(2014-11-28 as date))
   +- SubqueryAlias local.fifa.male_players_partition_fifa_update_date
      +- RelationV2[player_id#3033, player_url#3034, fifa_version#3035, fifa_update#3036, fifa_update_date#3037, short_name#3038, long_name#3039, player_positions#3040, overall#3041, potential#3042, value_eur#3043, wage_eur#3044, age#3045, dob#3046, height_cm#3047, weight_kg#3048, league_id#3049, league_name#3050, league_level#3051, club_team_id#3052, club_name#3053, club_position#3054, club_jersey_number#3055, club_loaned_from#3056, club_joined_date#3057, ... 85 more fields] local.fifa.male_players_partition_fifa_update_date

== Analyzed Logical Plan ==
fifa_update_date: date, club_name: string
Project [fifa_update_date#3037, club_name#3053]
+- Filter (fifa_update_date#3037 = cast(2014-11-28 as date))
   +- SubqueryAlias local.fifa.male_players_partition_fifa_update_date
      +- RelationV

---

In [59]:
viewdf(df_select_filter, limit=3)

,fifa_update_date,club_name
0,2014-11-28,FC Barcelona
1,2014-11-28,Real Madrid
2,2014-11-28,FC Bayern München


time: 45.7 ms (started: 2026-07-22 20:51:16 +00:00)


# Table schema

In [24]:
df.printSchema()

root
 |-- player_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- league_id: integer (nullable = true)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = true)
 |-- club_team_id: integer (nullable = true)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nullable = true)
 |-- club_loane

# Finding the best partition collumn
Data distribution, cardinallity

In [35]:
%%sql 
SELECT fifa_update_date, count(*) as count
FROM local.fifa.male_players 
GROUP BY fifa_update_date
ORDER BY count DESC;

,fifa_update_date,count
0,2021-09-25,37497
1,2022-05-19,37491
2,2020-09-23,37286
3,2020-06-17,36899
4,2018-07-19,36046
5,2022-05-27,19870
6,2020-08-12,19578
7,2022-01-11,19421
8,2021-12-16,19419
9,2021-12-20,19419


time: 294 ms (started: 2026-07-22 19:38:56 +00:00)


In [45]:
%%sql
WITH summary AS (
    SELECT
        COUNT(*) AS total_rows,
        COUNT(fifa_update_date) AS non_null_rows,
        COUNT(*) - COUNT(fifa_update_date) AS null_rows,
        COUNT(DISTINCT fifa_update_date) AS distinct_values
    FROM local.fifa.male_players
)
SELECT
    format_number(total_rows, 0) AS total_rows,
    format_number(non_null_rows, 0) AS non_null_rows,
    format_number(null_rows, 0) AS null_rows,
    format_number(distinct_values, 0) AS distinct_values,
    concat(
        format_number(distinct_values * 100.0 / total_rows, 4),
        '%'
    ) AS cardinality_pct,
    format_number(
        total_rows * 1.0 / NULLIF(distinct_values, 0),
        2
    ) AS average_rows_per_value
FROM summary;

,total_rows,non_null_rows,null_rows,distinct_values,cardinality_pct,average_rows_per_value
0,"10,003,590","10,003,590",0,553,0.0055%,"18,089.67"


time: 462 ms (started: 2026-07-22 19:55:37 +00:00)


In [32]:
%%sql
SELECT club_team_id, count(*) as count
FROM local.fifa.male_players 
GROUP BY club_team_id
ORDER BY count DESC;

,club_team_id,count
0,NaN,116395
1,7.0,18398
2,9.0,18395
3,19.0,18386
4,1.0,18377
5,1799.0,18377
6,18.0,18372
7,11.0,18351
8,95.0,18335
9,483.0,18302


time: 530 ms (started: 2026-07-22 19:35:40 +00:00)


## Listing the table

In [7]:
%%sql 
SELECT * 
FROM local.fifa.male_players 
LIMIT 3;

,player_id,player_url,fifa_version,fifa_update,fifa_update_date,short_name,long_name,player_positions,overall,potential,value_eur,wage_eur,age,dob,height_cm,weight_kg,league_id,league_name,league_level,club_team_id,club_name,club_position,club_jersey_number,club_loaned_from,club_joined_date,club_contract_valid_until_year,nationality_id,nationality_name,nation_team_id,nation_position,nation_jersey_number,preferred_foot,weak_foot,skill_moves,international_reputation,work_rate,body_type,real_face,release_clause_eur,player_tags,player_traits,pace,shooting,passing,dribbling,defending,physic,attacking_crossing,attacking_finishing,attacking_heading_accuracy,attacking_short_passing,attacking_volleys,skill_dribbling,skill_curve,skill_fk_accuracy,skill_long_passing,skill_ball_control,movement_acceleration,movement_sprint_speed,movement_agility,movement_reactions,movement_balance,power_shot_power,power_jumping,power_stamina,power_strength,power_long_shots,mentality_aggression,mentality_interceptions,mentality_positioning,mentality_vision,mentality_penalties,mentality_composure,defending_marking_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_diving,goalkeeping_handling,goalkeeping_kicking,goalkeeping_positioning,goalkeeping_reflexes,goalkeeping_speed,ls,st,rs,lw,lf,cf,rf,rw,lam,cam,ram,lm,lcm,cm,rcm,rm,lwb,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk,player_face_url
0,253896,/player/253896/jorge-echeverria/210014,21,14,2020-11-27,J. Echeverría,Jorge Eliézer Echeverría Montilva,"RM, CAM",54,62,240000,500,20,2000-02-13,174,70,2019,Primera Division,1,110989,Caracas,SUB,22,None,2016-01-01,2024,61,Venezuela,NaN,None,NaN,Right,2,2,1,High/Low,Lean (170-185),No,564000,None,None,72.0,40.0,45.0,52.0,26.0,40.0,60,39,39,41,37,50,52,39,40,48,71,72,72,49,66,39,53,83,26,40,20,20,57,39,40,51,27,25,22,6,14,8,11,9,NaN,46+2,46+2,46+2,52,49,49,49,52,48+2,48+2,48+2,53+2,44+2,44+2,44+2,53+2,45+2,36+2,36+2,36+2,45+2,43+2,30+2,30+2,30+2,43+2,14+2,https://cdn.sofifa.net/players/253/896/21_120.png
1,253963,/player/253963/wei-ren/210014,21,14,2020-11-27,Ren Wei,任威,"CF, CAM",54,63,240000,2000,23,1997-04-09,184,77,2012,Super League,1,112978,Hebei CFFC,RES,33,None,2019-08-01,2022,155,China PR,NaN,None,NaN,Right,3,2,1,Medium/Medium,Lean (170-185),No,420000,None,None,64.0,55.0,44.0,54.0,19.0,53.0,38,59,53,44,53,52,53,36,36,55,62,65,60,51,58,48,62,58,60,51,29,19,53,55,64,40,16,13,13,5,13,12,6,9,NaN,54+2,54+2,54+2,53,54,54,54,53,52+2,52+2,52+2,52+2,46+2,46+2,46+2,52+2,38+2,35+2,35+2,35+2,38+2,36+2,32+2,32+2,32+2,36+2,13+2,https://cdn.sofifa.net/players/253/963/21_120.png
2,254223,/player/254223/joshua-render/210014,21,14,2020-11-27,J. Render,Joshua Render,GK,54,68,220000,900,19,2000-09-01,183,70,14,Championship,2,1807,Sheffield Wednesday,RES,31,None,2019-07-01,2021,14,England,NaN,None,NaN,Right,3,1,1,Medium/Medium,Lean (170-185),No,556000,None,None,NaN,NaN,NaN,NaN,NaN,NaN,14,8,13,21,9,9,10,10,20,11,18,33,35,47,48,41,61,16,52,8,22,7,4,38,13,31,9,12,11,58,59,55,49,50,26.0,19+2,19+2,19+2,17,19,19,19,17,20+2,20+2,20+2,18+2,20+2,20+2,20+2,18+2,17+2,19+2,19+2,19+2,17+2,17+2,20+2,20+2,20+2,17+2,53+2,https://cdn.sofifa.net/players/254/223/21_120.png


time: 832 ms (started: 2026-07-22 19:20:41 +00:00)


In [10]:
%%sql
SELECT COUNT(*) AS data_file_count
FROM local.demo.people.data_files;


,data_file_count
0,4


time: 67.8 ms (started: 2026-07-22 19:21:11 +00:00)


In [12]:
%%sql
SELECT
    file_path,
    file_format,
    record_count,
    file_size_in_bytes
FROM local.demo.people.data_files;

,file_path,file_format,record_count,file_size_in_bytes
0,file:/home/jovyan/work/data/datalake/jupyter-s...,PARQUET,1,686
1,file:/home/jovyan/work/data/datalake/jupyter-s...,PARQUET,1,679
2,file:/home/jovyan/work/data/datalake/jupyter-s...,PARQUET,1,686
3,file:/home/jovyan/work/data/datalake/jupyter-s...,PARQUET,2,689


time: 37.9 ms (started: 2026-07-22 19:21:30 +00:00)


In [14]:
%%sql
-- To see the partition layout:
SELECT *
FROM local.demo.people.partitions;

,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
0,5,4,2740,0,0,0,0,2026-07-22 19:12:07.218,5161294362761053818


time: 47.2 ms (started: 2026-07-22 19:21:56 +00:00)


In [22]:
%%sql
-- To see which sort-order ID each data file uses:
SELECT
    file_path,
    sort_order_id
FROM local.demo.people.files;

,file_path,sort_order_id
0,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/demo/people/data/00000-3-e1521d21-9440-4dc9-b443-da335d02785c-0-00001.parquet,0
1,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/demo/people/data/00001-4-e1521d21-9440-4dc9-b443-da335d02785c-0-00001.parquet,0
2,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/demo/people/data/00000-0-3fe7cc06-f41f-4b9e-8fa1-f73d68d85244-0-00001.parquet,0
3,file:/home/jovyan/work/data/datalake/jupyter-spark-4.1/spark-4.1/iceberg/warehouse/demo/people/data/00001-1-3fe7cc06-f41f-4b9e-8fa1-f73d68d85244-0-00001.parquet,0


time: 40.1 ms (started: 2026-07-22 19:26:59 +00:00)


In [9]:
%run_nb spark-show

Job Id ▾,Description,Submitted,Duration,Stages: Succeeded/Total,Tasks (for all stages): Succeeded/Total
6,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:20:43,14 ms,1/1,1/1
5,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:20:43,10 ms,1/1,1/1
4,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:20:43,28 ms,1/1,1/1
3,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:20:42,19 ms,1/1,1/1
2,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:20:42,31 ms,1/1 (1 skipped),1/1 (1 skipped)
1,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:20:42,54 ms,1/1,1/1
0,toPandas at /home/jovyan/work/spark/lib.py:82 toPandas at /home/jovyan/work/spark/lib.py:82,2026/07/22 19:20:42,0.5 s,1/1,1/1


Stage Id ▾,Description,Submitted,Duration,Tasks: Succeeded/Total,Input,Output,Shuffle Read,Shuffle Write
7,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/22 19:20:43,12 ms,1/1,14.0 KiB,,,
6,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/22 19:20:43,8 ms,1/1,,,,
5,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thread.run(Thread.java:1583),2026/07/22 19:20:43,19 ms,1/1,14.0 KiB,,,
4,toPandas at /home/jovyan/work/spark/lib.py:82 +details org.apache.spark.sql.classic.Dataset.collectToPython(Dataset.scala:2081) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method) java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:75) java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:52) java.base/java.lang.reflect.Method.invoke(Method.java:580) py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244) py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374) py4j.Gateway.invoke(Gateway.java:282) py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132) py4j.commands.CallCommand.execute(CallCommand.java:79) py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184) py4j.ClientServerConnection.run(ClientServerConnection.java:108) java.base/java.lang.Thr

Version,4.1.2
Master,local[2]
AppName,main


time: 141 ms (started: 2026-07-22 19:20:43 +00:00)
